# Optimization & Signal Comparison

Parameter optimization (walk-forward) and individual signal performance analysis.

> **Prerequisite:** Run `main_mean_reversion.ipynb` first to generate bridge data.


In [ ]:
# ============================================================================
# LOAD BRIDGE DATA — from core backtest notebook
# ============================================================================
import gc
gc.collect()

%load_ext autoreload
%autoreload 2

import sys, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.precision', 4)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Resolve project root
project_root = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from strategy_config import ConfigLoader
from backtest.engine import BacktestEngine, BacktestConfig, calculate_rolling_sharpe, calculate_underwater_curve
from backtest.optimizer import ParameterOptimizer, OptimizationConfig, plot_wf_results

# Load bridge data
_bridge_path = project_root / 'data' / 'snapshots' / 'notebook_bridge' / 'core_results.pkl'
with open(_bridge_path, 'rb') as f:
    _bridge = pickle.load(f)

results = _bridge['results']
bt_config = _bridge['bt_config']
config = ConfigLoader(Path(_bridge['config_path']))
price_df = _bridge['price_df']
signal_df = _bridge['signal_df']
volume_df = _bridge['volume_df']
zscore_df = _bridge['zscore_df']
analysis_df = _bridge['analysis_df']
all_data = _bridge['all_data']
all_individual_signals = _bridge['all_individual_signals']
mean_reverting_symbols = _bridge['mean_reverting_symbols']
common_index = _bridge['common_index']

del _bridge
gc.collect()

print('✓ Bridge data loaded successfully')
print(f'  Symbols: {len(signal_df.columns)}')
print(f'  Date range: {signal_df.index.min()} → {signal_df.index.max()}')
print(f'  Backtest trades: {results.total_trades}')


## 6. Parameter Optimization

Use walk-forward analysis to find optimal parameters.

In [ ]:
# Configure optimization from config (toggle-aware)
optimization_enabled = config.get('optimization.enabled', True)

if not optimization_enabled:
    print("⏭️  Optimization DISABLED in config.yaml (optimization.enabled: false)")
    print("   Using current config parameters for backtest.")
    opt_results = None
else:
    opt_config = config.to_optimization_config()

    # Pass base backtest config so optimizer inherits sizing/log-return settings
    optimizer = ParameterOptimizer(opt_config, base_backtest_config=bt_config)

    print("Optimization configuration:")
    print(f"  Method: {opt_config.method}")
    print(f"  Trials: {opt_config.n_trials}  |  Parallel jobs: {opt_config.n_jobs}")
    print(f"  Train period: {opt_config.train_period_days} days")
    print(f"  Test period: {opt_config.test_period_days} days")
    print(f"  Step size: {opt_config.step_days} days")
    print(f"  Objective: {opt_config.objective_metric}")
    if opt_config.method == 'bayesian':
        print(f"  Bayesian ranges:")
        print(f"    entry_threshold: {list(opt_config.bayesian_entry_range)}")
        print(f"    exit_threshold: {list(opt_config.bayesian_exit_range)}")
        print(f"    stop_loss: {opt_config.bayesian_stop_loss_choices}")
        print(f"    take_profit: {opt_config.bayesian_take_profit_choices}")
        print(f"    max_holding: {opt_config.bayesian_max_holding_choices}")
    else:
        print(f"  Grid ranges:")
        print(f"    entry_threshold: {opt_config.entry_threshold_range}")
        print(f"    exit_threshold: {opt_config.exit_threshold_range}")
        print(f"    stop_loss: {opt_config.stop_loss_range}")
        print(f"    take_profit: {opt_config.take_profit_range}")
    print(f"\n  Base sizing config inherited:")
    print(f"    position_size_method: {bt_config.position_size_method}")
    print(f"    use_log_returns: {bt_config.use_log_returns}")

In [ ]:
if optimization_enabled:
    # Create signal generator function for optimizer
    def signal_generator_fn(params):
        """Generate signals (independent of backtest params)"""
        return signal_df

    # Run optimization
    print("Running walk-forward optimization...")
    print("This may take several minutes...\n")

    opt_results = optimizer.walk_forward_optimization(
        price_df,
        signal_generator_fn,
        volume_df,
        exit_signal_data=zscore_df
    )

    print("\n" + "="*50)
    print("Optimization complete!")
    print("="*50)
    print("\nSummary:")
    for key, value in opt_results.summary().items():
        print(f"{key:25s}: {value}")
else:
    print("⏭️  Optimization skipped.")

In [ ]:
# Plot optimization results (if optimization was enabled)
if opt_results is not None:
    plot_wf_results(opt_results)
else:
    print("No optimization results to plot.")

In [ ]:
# Parameter stability analysis (if optimization was enabled)
if opt_results is not None:
    print("\nParameter Frequency (across all periods):")
    print("="*50)
    for param, count in sorted(opt_results.best_params_frequency.items(), key=lambda x: x[1], reverse=True):
        print(f"{param:30s}: {count} times")
else:
    print("No optimization results for parameter stability analysis.")

In [ ]:
# Compare individual signals vs composite
# Note: zscore is on z-score scale (can cross 2.0), but bollinger/rsi are [-1, 0, +1]
# so we test those with a lower threshold (0.5) to see their standalone performance
# Diagnostic signals (kalman_mean, kalman_std, expected_return, time_to_reversion) are skipped
# Phase B.4: All signals now inherit full risk management params (stop loss, trailing stop, etc.)

print("Testing individual signals...\n")

individual_results = {}
# Only backtest actual trade signals, skip diagnostic outputs
signal_types = ['zscore', 'bollinger', 'rsi_divergence', 'rsi_level']
diagnostic_keys = {'kalman_mean', 'kalman_std', 'expected_return', 'time_to_reversion'}

from dataclasses import asdict

for signal_type in signal_types:
    # Check that this signal exists in individual signals
    first_sym = next(iter(all_individual_signals))
    if signal_type not in all_individual_signals[first_sym]:
        continue
    
    # Extract individual signal for all symbols
    individual_signal_df = pd.DataFrame({
        symbol: signals[signal_type]
        for symbol, signals in all_individual_signals.items()
        if signal_type in signals
    })
    
    # Align with prices
    individual_signal_df = individual_signal_df.loc[common_index]
    
    # Use appropriate threshold: z-score scale for zscore, lower for binary signals
    if signal_type == 'zscore':
        threshold = bt_config.entry_threshold
        exit_th = bt_config.exit_threshold
    else:
        threshold = 0.5  # These are [-1, 0, +1] signals
        exit_th = 0.1
    
    # Inherit ALL risk management params from base config (Phase B.4)
    # Only override entry/exit thresholds per signal type
    base_dict = asdict(bt_config)
    base_dict['entry_threshold'] = threshold
    base_dict['exit_threshold'] = exit_th
    ind_bt_config = BacktestConfig(**base_dict)
    
    ind_engine = BacktestEngine(ind_bt_config)
    result = ind_engine.run_backtest(price_df, individual_signal_df, volume_df)
    individual_results[signal_type] = result
    
    ev_str = f" | EV/trade: {result.ev_per_trade*100:.3f}%" if hasattr(result, 'ev_per_trade') and result.ev_per_trade is not None else ""
    print(f"{signal_type:20s} - Return: {result.total_return*100:6.2f}% | Sharpe: {result.sharpe_ratio:5.2f} | Trades: {result.total_trades}{ev_str}")

# Composite uses the standard threshold
ev_str = f" | EV/trade: {results.ev_per_trade*100:.3f}%" if hasattr(results, 'ev_per_trade') and results.ev_per_trade is not None else ""
print(f"\n{'Composite (gated)':20s} - Return: {results.total_return*100:6.2f}% | Sharpe: {results.sharpe_ratio:5.2f} | Trades: {results.total_trades}{ev_str}")

In [ ]:
# Visualize strategy comparison
fig = go.Figure()

# Add traces for each strategy
for name, result in individual_results.items():
    fig.add_trace(go.Scatter(
        x=result.equity_curve.index,
        y=result.equity_curve,
        mode='lines',
        name=name,
        opacity=0.6
    ))

# Add composite
fig.add_trace(go.Scatter(
    x=results.equity_curve.index,
    y=results.equity_curve,
    mode='lines',
    name='Composite',
    line=dict(width=3, color='purple')
))

fig.update_layout(
    title='Strategy Comparison - Equity Curves',
    xaxis_title='Date',
    yaxis_title='Equity ($)',
    hovermode='x unified',
    height=600
)

fig.show()

# Free individual backtest objects (no longer needed after visualization)
del individual_results, ind_engine, ind_bt_config
import gc; gc.collect()

## 8. Summary & Next Steps

Key insights and recommendations for Phase 3 (ML Filter).

In [9]:
print("Phase 2 / 2.5 Summary:")
print("="*60)
print(f"\n1. Universe Analysis:")
print(f"   - Total stocks analyzed: {len(analysis_df)}")
print(f"   - Mean-reverting stocks: {len(mean_reverting_symbols)}")
print(f"   - Average Hurst exponent: {analysis_df['hurst'].mean():.3f}")

print(f"\n2. Strategy Performance:")
print(f"   - Total Return: {results.total_return*100:.2f}%")
print(f"   - Sharpe Ratio: {results.sharpe_ratio:.2f}")
print(f"   - Max Drawdown: {results.max_drawdown*100:.2f}%")
print(f"   - Win Rate: {results.win_rate*100:.2f}%")
print(f"   - Total Trades: {results.total_trades}")
print(f"   - Profit Factor: {results.profit_factor:.2f}")

# Phase 2.5 EV metrics
if hasattr(results, 'ev_per_trade') and results.ev_per_trade is not None:
    print(f"\n3. Expected Value (EV) Analysis:")
    print(f"   - EV/Trade (overall): {results.ev_per_trade*100:.4f}%")
    if hasattr(results, 'ev_per_trade_long') and results.ev_per_trade_long is not None:
        print(f"   - EV/Trade (long):    {results.ev_per_trade_long*100:.4f}%")
    if hasattr(results, 'ev_per_trade_short') and results.ev_per_trade_short is not None:
        print(f"   - EV/Trade (short):   {results.ev_per_trade_short*100:.4f}%")
    print(f"   - Sizing Method: {bt_config.position_size_method}")
    print(f"   - Log Returns: {bt_config.use_log_returns}")

try:
    if opt_results is not None:
        print(f"\n4. Optimized Strategy Performance:")
        print(f"   - Test Return: {opt_results.combined_test_results.total_return*100:.2f}%")
        print(f"   - Test Sharpe: {opt_results.combined_test_results.sharpe_ratio:.2f}")
        print(f"   - Stability Score: {opt_results.stability_score:.3f}")
    else:
        print(f"\n4. Optimization: (disabled in config)")
except (NameError, AttributeError):
    print(f"\n4. Optimization: (run optimization cells first)")

print(f"\n5. Best Signal Type:")
try:
    best_signal = max(individual_results.items(), key=lambda x: x[1].sharpe_ratio)
    print(f"   - {best_signal[0]}: Sharpe {best_signal[1].sharpe_ratio:.2f}")
except NameError:
    print(f"   - (run strategy comparison cells first)")

# Phase 2.5 config recap
print(f"\n6. Phase 2.5 Upgrades Active:")
print(f"   - Kalman Filter: {config.get('signals.kalman.use_kalman', True)}")
print(f"   - OU Prediction Gate: {config.get('signals.ou_prediction.use_predicted_return', True)}")
print(f"   - Log Prices (z-score): {config.get('signals.use_log_prices', True)}")
print(f"   - Log Returns (metrics): {config.get('backtest.use_log_returns', True)}")
print(f"   - Position Sizing: {config.get('backtest.position_size_method')}")

print("\n" + "="*60)

Phase 2 / 2.5 Summary:

1. Universe Analysis:
   - Total stocks analyzed: 294
   - Mean-reverting stocks: 216
   - Average Hurst exponent: 0.480

2. Strategy Performance:
   - Total Return: 12339.13%
   - Sharpe Ratio: 3.91
   - Max Drawdown: 3.96%
   - Win Rate: 60.81%
   - Total Trades: 23988
   - Profit Factor: 1.96

3. Expected Value (EV) Analysis:
   - EV/Trade (overall): 0.6373%
   - EV/Trade (long):    0.5578%
   - EV/Trade (short):   0.7709%
   - Sizing Method: volatility_scaled
   - Log Returns: True

4. Optimization: (disabled in config)

5. Best Signal Type:
   - (run strategy comparison cells first)

6. Phase 2.5 Upgrades Active:
   - Kalman Filter: True
   - OU Prediction Gate: True
   - Log Prices (z-score): True
   - Log Returns (metrics): True
   - Position Sizing: volatility_scaled



In [10]:
# Extract consensus parameters from optimizer results
import numpy as np
from collections import Counter

if opt_results is None:
    print("Optimizer was disabled — skipping parameter consensus analysis.")
    print("To enable, set optimization.enabled: true in config.yaml and rerun.")
else:
    periods = opt_results.walk_forward_results

    # Collect all parameters
    entries = [p.best_params['entry_threshold'] for p in periods]
    exits = [p.best_params['exit_threshold'] for p in periods]
    stops = [p.best_params.get('stop_loss_pct') for p in periods]
    takes = [p.best_params.get('take_profit_pct') for p in periods]
    holdings = [p.best_params.get('max_holding_days') for p in periods]

    print("=" * 70)
    print("OPTIMIZER PARAMETER CONSENSUS ANALYSIS (34 Walk-Forward Periods)")
    print("=" * 70)

    print(f"\n--- Entry Threshold ---")
    print(f"  Mean: {np.mean(entries):.3f}  |  Median: {np.median(entries):.3f}  |  Std: {np.std(entries):.3f}")
    print(f"  Range: [{min(entries):.3f}, {max(entries):.3f}]")
    print(f"  25th-75th pctl: [{np.percentile(entries, 25):.3f}, {np.percentile(entries, 75):.3f}]")
    print(f"  Current config: 1.500")

    print(f"\n--- Exit Threshold ---")
    print(f"  Mean: {np.mean(exits):.3f}  |  Median: {np.median(exits):.3f}  |  Std: {np.std(exits):.3f}")
    print(f"  Range: [{min(exits):.3f}, {max(exits):.3f}]")
    print(f"  25th-75th pctl: [{np.percentile(exits, 25):.3f}, {np.percentile(exits, 75):.3f}]")
    print(f"  Current config: 0.817")

    print(f"\n--- Stop Loss ---")
    stop_counts = Counter(str(s) for s in stops)
    print(f"  Distribution: {dict(stop_counts)}")
    numeric_stops = [s for s in stops if s is not None]
    if numeric_stops:
        print(f"  Numeric mean: {np.mean(numeric_stops):.3f}  |  Median: {np.median(numeric_stops):.3f}")
    print(f"  Current config: 0.08")

    print(f"\n--- Take Profit ---")
    tp_counts = Counter(str(t) for t in takes)
    print(f"  Distribution: {dict(tp_counts)}")
    numeric_tp = [t for t in takes if t is not None]
    if numeric_tp:
        print(f"  Numeric mean: {np.mean(numeric_tp):.3f}  |  Median: {np.median(numeric_tp):.3f}")
    print(f"  Current config: 0.15")

    print(f"\n--- Max Holding Days ---")
    hold_counts = Counter(str(h) for h in holdings)
    print(f"  Distribution: {dict(hold_counts)}")
    numeric_holds = [h for h in holdings if h is not None]
    if numeric_holds:
        print(f"  Numeric mean: {np.mean(numeric_holds):.1f}  |  Median: {np.median(numeric_holds):.0f}")
    print(f"  Current config: 30")

    # Recommended consensus
    print(f"\n{'=' * 70}")
    print(f"RECOMMENDED CONSENSUS PARAMETERS")
    print(f"{'=' * 70}")
    entry_consensus = round(np.median(entries), 2)
    exit_consensus = round(np.median(exits), 2)
    stop_mode = Counter(stops).most_common(1)[0][0]
    tp_mode = Counter(takes).most_common(1)[0][0]
    hold_mode = Counter(holdings).most_common(1)[0][0]

    print(f"  entry_threshold:  {entry_consensus}  (median, current: 1.50)")
    print(f"  exit_threshold:   {exit_consensus}  (median, current: 0.82)")
    print(f"  stop_loss_pct:    {stop_mode}  (mode, current: 0.08)")
    print(f"  take_profit_pct:  {tp_mode}  (mode, current: 0.15)")
    print(f"  max_holding_days: {hold_mode}  (mode, current: 30)")

    # OOS performance per period
    test_sharpes = [p.test_results.sharpe_ratio for p in periods]
    test_returns = [p.test_results.total_return * 100 for p in periods]
    pos_periods = sum(1 for s in test_sharpes if s > 0)
    print(f"\n{'=' * 70}")
    print(f"PERIOD-LEVEL OUT-OF-SAMPLE PERFORMANCE")
    print(f"{'=' * 70}")
    print(f"  Periods with positive Sharpe: {pos_periods}/{len(test_sharpes)} ({pos_periods/len(test_sharpes)*100:.0f}%)")
    print(f"  Mean test Sharpe: {np.mean(test_sharpes):.3f}  |  Median: {np.median(test_sharpes):.3f}")
    print(f"  Mean test return: {np.mean(test_returns):.2f}%  |  Median: {np.median(test_returns):.2f}%")
    print(f"  Best period Sharpe: {max(test_sharpes):.3f}  |  Worst: {min(test_sharpes):.3f}")

Optimizer was disabled — skipping parameter consensus analysis.
To enable, set optimization.enabled: true in config.yaml and rerun.
